# LEVELS Benchmark — Level 3: Discovery

Hypothesis-level, DiscoveryBench-style tasks: the agent must analyze real data and state a
finding as a structured hypothesis, scored against `ground_truth_hypothesis` facets
(`context`/`variables`/`relationship`) by a faceted LLM judge (`score_level3`). Every ground
truth below was verified directly against `data/oceans_11/*.db` with sqlite3 (see chat
history / `BENCHMARK_METHODOLOGY.md` for the verification queries) rather than assumed, since
a wrong ground truth would make this an unfalsifiable benchmark.

Shared harness code (scoring, parallel runner, reports) lives in `levels_bench/`, and the task
definitions in `levels_bench/tasks/`. Each (model, task) pair runs as an independent job in a
pool of worker processes, so `max_workers` runs proceed at once. Every finished result is
appended to `levels_benchmark_results/<run_id>_<label>.partial.jsonl` as it completes. If a run
is interrupted, call `run_benchmark(..., resume=True)` to run only what's missing.

## Setup

In [ ]:
# Run once per environment (not needed on every run):
# %pip install -q -r requirements.txt
# %pip install -q --no-build-isolation -e ../..

In [ ]:
from levels_bench import init, run_benchmark, flatten, results_to_dataframe, load_results, reports, debug_task
from levels_bench.tasks import LEVEL3_TASKS, get_task

cfg = init(
    # models_to_run=["aws-gov.gpt-5.4"],   # default: every model in levels_bench.config.POSSIBLE_MODELS
    # judge_models=[...],                   # default: same list; judged scores are averaged across them
    max_workers=4,                          # concurrent agent runs; raise if the portal's rate limit allows
    task_timeout_sec=180,                   # per agent turn
    # skip_network=True,                    # skip tasks that download from oceans11.lanl.gov
)

In [ ]:
reports.task_table(LEVEL3_TASKS)

## Run

In [ ]:
l3_results = flatten(run_benchmark(LEVEL3_TASKS, "level3_discovery"))
# To analyze an earlier run without re-running it:
# l3_results = load_results("level3_discovery")
l3_df = results_to_dataframe(l3_results)
l3_df

## Results

In [ ]:
reports.model_summary(l3_df)

In [ ]:
reports.level3_facets(l3_df)

In [ ]:
reports.failures(l3_df)

In [ ]:
reports.timing_breakdown(l3_results)

## Inspect / debug

Transcripts are recorded for every run. `debug_task` reruns one task in this process with live output.

In [ ]:
# reports.transcript(l3_results, "L3_nif_density_trend")
# tr = debug_task(get_task("L3_nif_density_trend"), model="anthropic.claude-sonnet-4-5-20250929-v1:0")